In [11]:
import ccxt
import pandas as pd
import ta
import numpy as np
import vectorbt as vbt
import warnings
warnings.filterwarnings('ignore')

In [12]:
## 전략부분까찌만
def rsi_hs(start):
    df = vbt.CCXTData.download('BTC/USDT', timeframe='1h', start = start).get()
    df['rsi'] = ta.momentum.RSIIndicator(df['Close']).rsi()
    df['3ma_rsi'] = ta.trend.EMAIndicator(df['rsi'], 3).ema_indicator()
    df['signal_rsi'] = np.where(df['rsi'] > df['3ma_rsi'], 1,-1)

    df['Close_h'] =  round(((df['Open'] + df['High'] + df['Low'] + df['Close'])/4),2)
    df['50ma_h'] = ta.trend.SMAIndicator(df['Close_h'], 50).sma_indicator()
    df['signal_ma'] = np.where(df['Close'] > df['50ma_h'], 1,-1)
    
    cond_go = [ (df['signal_rsi'] == 1) & (df['signal_ma'] == 1), 
               (df['signal_rsi'] == -1) & (df['signal_ma'] == -1)  ]
    df['go'] =np.select(cond_go, [1, -1], 0)
    
    pf = vbt.Portfolio.from_signals(df['Close'], 
                                entries = (df['go'] == 1), 
                                exits = (df['go'] == -1), 
                                fees = 0.0006,
                                tp_stop = 0.09,
                                sl_stop = 0.03,
                                short_entries = (df['go'] == -1),
                                short_exits = (df['go'] == 1)                                                         
                               )
    
    return pf.stats()['Total Return [%]'], pf.stats()['Win Rate [%]'], pf.stats()['Max Drawdown [%]']

In [13]:
def bb_psar(start):
    df = vbt.CCXTData.download('BTC/USDT', timeframe='4h', start = start).get()
    df['bb_h'] = vbt.ta('BollingerBands').run(df['Close']).bollinger_hband
    df['bb_l'] = vbt.ta('BollingerBands').run(df['Close']).bollinger_lband
    df['signal_bb'] = 0
        
    for current in range(1, len(df.index)):
        previous = current - 1
        if df.iat[current, 1] > df.iat[current, 5]:
            if df.iat[previous, 1] < df.iat[previous, 5]:
                df.iat[current, 7] = 1
        elif df.iat[current, 2] < df.iat[current, 6]:
            if df.iat[previous, 2] > df.iat[previous, 6] :
                df.iat[current, 7] = -1
            
    df['psar'] = ta.trend.PSARIndicator(df['High'], df['Low'], df['Close']).psar()
    df['signal_psar'] = np.where(df['Close'] > df['psar'], 1, -1)
    
    cond_go = [ (df['signal_bb'] == 1) & (df['signal_psar'] == 1), 
               (df['signal_bb'] == -1) & (df['signal_psar'] == -1)  ]
    df['go'] =np.select(cond_go, [1, -1], 0)
    
    pf = vbt.Portfolio.from_signals(df['Close'], 
                                entries = (df['go'] == 1), 
                                exits = (df['go'] == -1), 
                                fees = 0.0006,
                                tp_stop = 0.09,
                                sl_stop = 0.03,
                                short_entries = (df['go'] == -1),
                                short_exits = (df['go'] == 1)                                                         
                               )
    
    return pf.stats()['Total Return [%]'], pf.stats()['Win Rate [%]'], pf.stats()['Max Drawdown [%]']


In [14]:
def engulfing_rsi(start):
    df = vbt.CCXTData.download('BTC/USDT', timeframe='15m', start = start ).get()
    df['signal_engulfing'] = 0
    for current in range(1, len(df.index)):
        previous = current - 1
        if df.iat[current, 3] > df.iat[previous, 0]:
            if df.iat[current, 0] < df.iat[previous, 3]:
                df.iat[current, 5] = 1
        elif df.iat[current, 3] < df.iat[previous, 0]:
            if df.iat[current, 0] > df.iat[previous, 3]:
                df.iat[current, 5] = -1

    df['rsi'] = ta.momentum.RSIIndicator(df['Close']).rsi()
    cond_rsi = [ df['rsi'] < 30, df['rsi'] > 60 ]
    df['signal_rsi'] = np.select(cond_rsi, [1,-1], 0)
    
    cond_go = [ (df['signal_engulfing'] == 1) & (df['signal_rsi'] == 1), 
               (df['signal_engulfing'] == -1) & (df['signal_rsi'] == -1)  ]
    df['go'] =np.select(cond_go, [1, -1], 0)
    
    pf = vbt.Portfolio.from_signals(df['Close'], 
                                entries = (df['go'] == 1), 
                                exits = (df['go'] == -1), 
                                fees = 0.0006,
                                tp_stop = 0.09,
                                sl_stop = 0.03,
                                short_entries = (df['go'] == -1),
                                short_exits = (df['go'] == 1)                                                         
                               )
    
    return pf.stats()['Total Return [%]'], pf.stats()['Win Rate [%]'], pf.stats()['Max Drawdown [%]']

In [15]:
list_strategy = [rsi_hs, bb_psar, engulfing_rsi]
empty = []
for strategy in list_strategy:
    (total_return, win_rate, max_drawdown) = strategy('6 months ago')
    dct = {}
    dct['name'] = strategy.__name__
    dct['total_return'] = total_return
    dct['win_rate'] = win_rate
    dct['max_drawdown'] = max_drawdown
    empty.append(dct)

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

In [17]:
pd.DataFrame(empty).sort_values('name', ascending=False)

,name,total_return,win_rate,max_drawdown
0,rsi_hs,-43.101544,17.891374,50.272796
2,engulfing_rsi,66.443778,54.117647,37.946148
1,bb_psar,-10.407768,37.777778,29.739746
